# Course 1 Workshop: Spatial Analytics for DDC Dengue Surveillance

Welcome! This notebook is your **hands-on workspace** for the workshop.

**How to use this notebook:**
1. Read the scenario and instructions in each section
2. Write your SQL in the empty code cells
3. Press **Shift+Enter** to run the cell and see results
4. If you get stuck, expand the **Hint** or **Answer** sections

**Tables available:**

| Table | Rows | Key Columns |
|-------|------|-------------|
| `dengue_cases` | 30 | case_id, patient_age, gender, infection_date, severity, geog |
| `hospitals` | 15 | id, name, province, district, geog |
| `schools` | 10 | id, name, district, geog |
| `province_boundaries` | 3 | province_code, province_name, boundary |

All spatial columns use **GEOGRAPHY** type (distances in meters).

In [1]:
# Run this cell first -- it loads the helper functions
from ddc_helpers import run_query, show_on_map, show_buffers, show_heatmap, explain_query

## Warm-Up: Explore the Data

Before we start the exercises, let's look at what we have.

In [5]:
# How many cases, by severity?
run_query("""
SELECT severity, COUNT(*) AS case_count,
    ROUND(AVG(patient_age), 1) AS avg_age
FROM dengue_cases
GROUP BY severity
ORDER BY case_count DESC
""");

,severity,case_count,avg_age
0,DF,20,25.8
1,DHF,7,15.6
2,DSS,3,19.7


In [6]:
run_query("""
SELECT 'Case ' || case_id AS label, severity, patient_age,
    ST_Y(geog) AS lat, ST_X(geog) AS lon
FROM dengue_cases
""");

,label,severity,patient_age,lat,lon
0,Case 1,DF,8,13.7230,100.5530
1,Case 2,DF,25,13.7210,100.5560
2,Case 3,DHF,12,13.7250,100.5510
3,Case 4,DF,45,13.7200,100.5580
4,Case 5,DSS,6,13.7240,100.5540
5,Case 6,DF,33,13.7220,100.5520
6,Case 7,DHF,18,13.7190,100.5570
7,Case 8,DF,9,13.7260,100.5500
8,Case 9,DF,55,13.7235,100.5550
9,Case 10,DHF,14,13.7215,100.5525


In [3]:
# Where are the cases on the map?
show_on_map("""
SELECT 'Case ' || case_id AS label, severity, patient_age,
    ST_Y(geog) AS lat, ST_X(geog) AS lon
FROM dengue_cases
""", lat_col="lat", lon_col="lon", color="red",
    title="All 30 Dengue Cases")

In [7]:
# Where are the hospitals?
show_on_map("""
SELECT name, province, district,
    ST_Y(geog) AS lat, ST_X(geog) AS lon
FROM hospitals
""", lat_col="lat", lon_col="lon", color="blue",
    title="15 Hospitals")

---

# Exercise 1: Proximity Analysis (10 min)

## Scenario

Your supervisor asks:

> *"Which 3 hospitals are closest to the Khlong Toei dengue cluster? How far away are they?"*

The Khlong Toei cluster is centered at **latitude 13.72, longitude 100.55**.

## Your Task

1. Create a GEOGRAPHY point for the cluster center
2. Calculate distance in meters from this point to every hospital
3. Return: hospital name, distance in meters (rounded), sorted closest first
4. Limit to top 3

**Key functions:**
- `ST_GeographyFromText('POINT(longitude latitude)')` -- note: longitude first!
- `ST_Distance(geog1, geog2)` -- returns meters
- `ROUND(value, 0)` -- round to whole number

In [ ]:
# Exercise 1: Write your query here
run_query("""



""")

<details>
<summary><b>Hint</b></summary>

```sql
SELECT
    h.name,
    ROUND(ST_Distance(
        h.geog,
        ST_GeographyFromText('POINT(___ ___)')  -- fill in lon, lat
    ), 0) AS distance_meters
FROM hospitals h
ORDER BY ___
LIMIT ___
```

Remember: WKT is `POINT(longitude latitude)` -- longitude comes first!

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT
    h.name AS hospital_name,
    ROUND(ST_Distance(
        h.geog,
        ST_GeographyFromText('POINT(100.55 13.72)')
    ), 0) AS distance_meters
FROM hospitals h
ORDER BY distance_meters ASC
LIMIT 3
```

**Expected results:**
1. Charoen Krung Pracharak Hospital: ~5,500m
2. Taksin Hospital: ~7,000m
3. King Chulalongkorn Memorial: ~7,200m

**Discussion:** If the 3 nearest hospitals are all more than 5 km away,
DDC should consider deploying mobile treatment units closer to the cluster.

</details>

---

# Exercise 2: Risk Scoring (10 min)

## Scenario

DDC needs to prioritize which schools to notify. Your team uses a **weighted risk score**:

| Severity | Radius | Points per case |
|----------|--------|-----------------|
| DSS (Dengue Shock Syndrome) | within 500m | **3 points** |
| DHF (Dengue Hemorrhagic Fever) | within 1 km | **2 points** |
| DF (Dengue Fever) | within 2 km | **1 point** |

## Your Task

1. For each school, score nearby cases using the table above
2. Return: school_name, total_risk_score, dss_cases, dhf_cases, df_cases
3. Order by risk score (highest first)

**Key functions:**
- `ST_DWithin(geog1, geog2, meters)` -- true if within N meters
- `CASE WHEN ... THEN ... ELSE ... END` -- conditional logic
- `SUM(CASE WHEN ...)` -- conditional aggregation

In [ ]:
# Exercise 2: Write your query here
run_query("""



""")

<details>
<summary><b>Hint</b></summary>

Use a LEFT JOIN with the widest radius (2km) to catch all cases, then use CASE to score:

```sql
SELECT
    s.name AS school_name,
    SUM(
        CASE
            WHEN d.severity = 'DSS' AND ST_DWithin(s.geog, d.geog, 500)  THEN 3
            WHEN d.severity = '___' AND ST_DWithin(s.geog, d.geog, ____) THEN 2
            WHEN d.severity = '___' AND ST_DWithin(s.geog, d.geog, ____) THEN 1
            ELSE 0
        END
    ) AS total_risk_score,
    ...
FROM schools s
LEFT JOIN dengue_cases d ON ST_DWithin(s.geog, d.geog, 2000)
GROUP BY s.name
ORDER BY total_risk_score DESC;
```

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT
    s.name AS school_name,
    SUM(
        CASE
            WHEN d.severity = 'DSS' AND ST_DWithin(s.geog, d.geog, 500)  THEN 3
            WHEN d.severity = 'DHF' AND ST_DWithin(s.geog, d.geog, 1000) THEN 2
            WHEN d.severity = 'DF'  AND ST_DWithin(s.geog, d.geog, 2000) THEN 1
            ELSE 0
        END
    ) AS total_risk_score,
    SUM(CASE WHEN d.severity = 'DSS' AND ST_DWithin(s.geog, d.geog, 500)  THEN 1 ELSE 0 END) AS dss_cases,
    SUM(CASE WHEN d.severity = 'DHF' AND ST_DWithin(s.geog, d.geog, 1000) THEN 1 ELSE 0 END) AS dhf_cases,
    SUM(CASE WHEN d.severity = 'DF'  AND ST_DWithin(s.geog, d.geog, 2000) THEN 1 ELSE 0 END) AS df_cases
FROM schools s
LEFT JOIN dengue_cases d ON ST_DWithin(s.geog, d.geog, 2000)
GROUP BY s.name
ORDER BY total_risk_score DESC
```

**Top results:** Wat Khlong Toei School (~14), Khlong Toei Wittaya (~13), Din Daeng Wittaya (~10)

**Discussion:** A school with score 0 might still need notification because:
- Outbreak could spread in their direction
- Students might live in the outbreak zone and commute
- Seasonal patterns suggest risk will increase

</details>

---

# Exercise 3: Heatmap Resolution (10 min)

## Scenario

DDC is preparing the weekly situation report. The team is debating which heatmap resolution to use.

Compare three GeoHash precisions for **Bangkok cases only**:

| Precision | Cell Size | Use Case |
|-----------|-----------|----------|
| 5 | ~5 km | National overview |
| 6 | ~1.2 km | City-level sitrep |
| 7 | ~150 m | Neighborhood fogging |

## Your Task

1. Filter cases to Bangkok only (use `province_boundaries` with `ST_Intersects`)
2. For each precision (5, 6, 7), count:
   - Distinct grid cells with cases
   - Total cases
   - Average cases per cell
3. Use `UNION ALL` to combine three queries

**Key functions:**
- `ST_Intersects(pb.boundary, d.geog)` -- filter to Bangkok
- `SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, N)` -- geohash at precision N
- `COUNT(DISTINCT ...)` -- count unique cells

In [ ]:
# Exercise 3: Write your query here
run_query("""



""")

<details>
<summary><b>Hint</b></summary>

Structure each of the three parts like:

```sql
SELECT
    5 AS precision,
    COUNT(DISTINCT SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(d.geog))), 1, 5)) AS cell_count,
    COUNT(*) AS total_cases,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT SUBSTR(..., 1, 5)), 1) AS avg_per_cell
FROM dengue_cases d
JOIN province_boundaries pb ON ST_Intersects(pb.boundary, d.geog)
WHERE pb.province_name = 'Bangkok'

UNION ALL

SELECT 6 AS precision, ...
```

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT
    5 AS precision,
    COUNT(DISTINCT SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(d.geog))), 1, 5)) AS cell_count,
    COUNT(*) AS total_cases,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(d.geog))), 1, 5)), 1) AS avg_per_cell
FROM dengue_cases d
JOIN province_boundaries pb ON ST_Intersects(pb.boundary, d.geog)
WHERE pb.province_name = 'Bangkok'

UNION ALL

SELECT
    6, COUNT(DISTINCT SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(d.geog))), 1, 6)),
    COUNT(*),
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(d.geog))), 1, 6)), 1)
FROM dengue_cases d
JOIN province_boundaries pb ON ST_Intersects(pb.boundary, d.geog)
WHERE pb.province_name = 'Bangkok'

UNION ALL

SELECT
    7, COUNT(DISTINCT SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(d.geog))), 1, 7)),
    COUNT(*),
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(d.geog))), 1, 7)), 1)
FROM dengue_cases d
JOIN province_boundaries pb ON ST_Intersects(pb.boundary, d.geog)
WHERE pb.province_name = 'Bangkok'

ORDER BY precision
```

**Insight:** Precision 6 is the sweet spot for weekly sitreps -- the two Bangkok clusters
separate into distinct cells. Precision 7 gives neighborhood detail for fogging maps.

</details>

### Bonus: Visualize your preferred resolution on a map

In [ ]:
# Visualize heatmap at precision 6 (or change to 5 or 7)
show_heatmap("""
SELECT
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(d.geog))), 1, 6) AS cell_id,
    COUNT(*) AS case_count,
    ST_AsText(ST_GeomFromGeoHash(
        SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(d.geog))), 1, 6)
    )) AS wkt
FROM dengue_cases d
JOIN province_boundaries pb ON ST_Intersects(pb.boundary, d.geog)
WHERE pb.province_name = 'Bangkok'
GROUP BY cell_id
ORDER BY case_count DESC
""", title="Bangkok Dengue Heatmap (Precision 6)")

---

# Exercise 4: Outbreak Timeline (10 min)

## Scenario

DDC leadership wants a monthly trend report for the 2024 dengue season:
- Which month was worst?
- Are severe cases increasing?

## Your Task

1. Group cases by month using `DATE_TRUNC`
2. For each month, calculate:
   - Total cases
   - Severe cases (DHF + DSS)
   - Average patient age (rounded to 1 decimal)
3. Order chronologically

**Key functions:**
- `DATE_TRUNC('month', infection_date)` -- truncates to first of month
- `SUM(CASE WHEN severity IN ('DHF','DSS') THEN 1 ELSE 0 END)` -- count severe

In [ ]:
# Exercise 4: Write your query here
run_query("""



""")

<details>
<summary><b>Hint</b></summary>

```sql
SELECT
    DATE_TRUNC('month', infection_date) AS report_month,
    COUNT(*)                            AS case_count,
    SUM(CASE WHEN severity IN (___) THEN 1 ELSE 0 END) AS severe_cases,
    ROUND(AVG(___), 1)                  AS avg_patient_age
FROM dengue_cases
GROUP BY ___
ORDER BY ___;
```

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT
    DATE_TRUNC('month', infection_date) AS report_month,
    COUNT(*)                            AS case_count,
    SUM(CASE WHEN severity IN ('DHF', 'DSS') THEN 1 ELSE 0 END) AS severe_cases,
    ROUND(AVG(patient_age), 1)          AS avg_patient_age
FROM dengue_cases
GROUP BY DATE_TRUNC('month', infection_date)
ORDER BY report_month
```

**Expected results:**

| report_month | case_count | severe_cases | avg_patient_age |
|---|---|---|---|
| 2024-06-01 | 3 | 1 | 15.0 |
| 2024-07-01 | 12 | 5 | 24.3 |
| 2024-08-01 | 10 | 3 | 26.2 |
| 2024-09-01 | 5 | 1 | 25.4 |

**Insight:** July is the peak -- classic rainy season outbreak curve.

</details>

### Bonus Challenge: Month-over-Month Change by Hexbin

Which grid cell had the sharpest month-over-month increase?

Use `LAG()` window function to compare consecutive months.

In [ ]:
# Bonus: Write your query here (optional)
run_query("""



""")

<details>
<summary><b>Bonus Answer</b></summary>

```sql
WITH monthly_hex AS (
    SELECT
        DATE_TRUNC('month', infection_date) AS report_month,
        SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_id,
        COUNT(*) AS case_count
    FROM dengue_cases
    GROUP BY report_month, cell_id
),
with_lag AS (
    SELECT
        report_month, cell_id, case_count,
        LAG(case_count, 1, 0) OVER (
            PARTITION BY cell_id ORDER BY report_month
        ) AS prev_month_cases,
        case_count - LAG(case_count, 1, 0) OVER (
            PARTITION BY cell_id ORDER BY report_month
        ) AS month_change
    FROM monthly_hex
)
SELECT * FROM with_lag
ORDER BY month_change DESC
LIMIT 5
```

The Khlong Toei cell shows the sharpest increase in July.

</details>

---

# Visualization: Putting It All Together

Run the cells below to see the combined outbreak picture.

In [ ]:
# Risk buffer zones across all provinces
show_buffers("""
SELECT
    pb.province_name,
    tier.risk_tier,
    tier.radius_meters,
    COUNT(DISTINCT d.case_id) AS case_count,
    ST_AsText(ST_Union(
        ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), tier.radius_meters / 111000.0)
    )) AS wkt
FROM dengue_cases d
JOIN province_boundaries pb ON ST_Intersects(pb.boundary, d.geog)
CROSS JOIN (
    SELECT 'RED' AS risk_tier, 500 AS radius_meters
    UNION ALL SELECT 'ORANGE', 1000
    UNION ALL SELECT 'YELLOW', 2000
) tier
GROUP BY pb.province_name, tier.risk_tier, tier.radius_meters
ORDER BY tier.radius_meters DESC
""")

In [ ]:
# Heatmap with all cases
show_heatmap("""
SELECT
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_id,
    COUNT(*) AS case_count,
    SUM(CASE WHEN severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_count,
    ST_AsText(ST_GeomFromGeoHash(
        SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6)
    )) AS wkt
FROM dengue_cases
GROUP BY cell_id
ORDER BY case_count DESC
""", title="Dengue Heatmap -- All Provinces")

---

# Cheat Sheet: Spatial Functions Reference

| Function | What It Does | Example |
|----------|-------------|---------|
| `ST_GeographyFromText(wkt)` | Create GEOGRAPHY from WKT | `ST_GeographyFromText('POINT(100.55 13.72)')` |
| `ST_Distance(a, b)` | Distance in meters | `ST_Distance(s.geog, d.geog)` |
| `ST_DWithin(a, b, m)` | True if within N meters | `ST_DWithin(s.geog, d.geog, 1000)` |
| `ST_Intersects(a, b)` | True if shapes overlap | `ST_Intersects(pb.boundary, d.geog)` |
| `ST_Buffer(geom, deg)` | Circle around point | `ST_Buffer(geom, 500.0/111000)` |
| `ST_Union(geom)` | Merge overlapping shapes | `ST_Union(ST_Buffer(...))` |
| `ST_Area(geom)` | Area (sq degrees) | `ST_Area(merged_zone) * 11988` for sq km |
| `ST_GeoHash(geom)` | Grid cell code | `SUBSTR(ST_GeoHash(geom), 1, 6)` |
| `ST_GeomFromGeoHash(h)` | Cell polygon | `ST_AsText(ST_GeomFromGeoHash('w4r0t'))` |
| `ST_X(geog)` / `ST_Y(geog)` | Extract lon / lat | For map display |
| `ST_AsText(geog)` | Convert to WKT text | For export / debugging |

**Conversion:** GEOGRAPHY to GEOMETRY: `ST_GeomFromText(ST_AsText(geog))`

**Degree-to-meter ratio** at Thai latitude: 1 degree ≈ 111,000 meters

---

# Scratch Pad

Use the cells below to experiment with your own queries.

In [9]:
# Your experiments here
run_query("""
SELECT 'Hello DDC!' AS message
""");

,message
0,Hello DDC!


In [ ]:
# More experiments
run_query("""

""")

In [ ]:
# Try show_on_map with your own query
# show_on_map("""  your SQL here  """, lat_col="lat", lon_col="lon")